# 06.02 · Chain-of-thought & reasoning models

**Chapter:** Prompt & Context Engineering · **Milestone:** M2

## Goals
- [ ] Direct answer vs chain-of-thought (CoT) vs self-consistency, measured
- [ ] Reasoning models: built-in thinking, thinking budgets, cost/latency trade-off
- [ ] Choose when reasoning is worth it

Runtime: ~4 min on a laptop with qwen3:1.7b (self-consistency and thinking are slow — that's the lesson).

## Resources
- [Chain-of-Thought Prompting (paper)](https://arxiv.org/abs/2201.11903) · [Self-Consistency (paper)](https://arxiv.org/abs/2203.11171) — free
- [OpenAI – Reasoning models guide](https://platform.openai.com/docs/guides/reasoning) · [Anthropic – Extended thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking) — free

Why CoT works: a transformer does a fixed amount of computation per token. Letting it write intermediate steps gives it more
tokens = more computation, and each step conditions on the previous ones. Reasoning models are trained (with RL, 04.04)
to do this well and at length.

In [ ]:
import re
import time
from collections import Counter

from aieng.llm import client

c, MODEL = client()
LOCAL = c.base_url.host == "localhost"
print(MODEL)

PROBLEMS = [  # (question, answer)
    ("A shop sells pens at 3 for $4. How many dollars do 18 pens cost?", 24),
    ("Ana has 5 boxes with 12 apples each. She gives away 17 apples. How many are left?", 43),
    ("A train leaves at 9:40 and arrives at 13:15. How many minutes is the trip?", 215),
    ("If 4 workers build a wall in 6 days, how many days do 3 workers need (same rate)?", 8),
    ("A shirt costs $40. It is discounted 25%, then the discounted price rises 10%. What is the final price in dollars?", 33),
    ("What is the sum of all integers from 1 to 40?", 820),
    ("A rectangle has perimeter 36 and length 11. What is its area?", 77),
    ("Luis is 3 times as old as his son. In 12 years he will be twice as old. How old is Luis now?", 36),
    ("How many minutes are there in 2.5 days?", 3600),
    ("A recipe needs 250 g of flour for 4 people. How many grams for 14 people?", 875),
    ("I have 3 red, 4 blue and 5 green marbles. I remove all blue ones and add twice as many red as I had. How many marbles now?", 14),
    ("A phone plan costs $15/month plus $0.10 per MB over 2000 MB. I used 2350 MB. What is my bill in dollars?", 50),
]

## 1. Methods

In [ ]:
def ask(prompt, temperature=0, think=False, **kw):
    extra = {} if not LOCAL else ({} if think else {"reasoning_effort": "none"})
    r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=temperature, **extra, **kw)
    return r.choices[0].message.content or "", r.usage.completion_tokens


def last_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(nums[-1]) if nums else None


def direct(q):
    out, toks = ask(f"{q}\nReply with only the final number, no words.")
    return last_number(out), toks


def cot(q, temperature=0):
    out, toks = ask(f"{q}\nThink step by step, then give the final line as 'Answer: <number>'.", temperature=temperature)
    m = re.search(r"Answer:\s*\$?(-?[\d,.]+)", out)
    return (float(m.group(1).replace(",", "").rstrip(".")) if m else last_number(out)), toks


def self_consistency(q, k=5):
    votes, total = [], 0
    for _ in range(k):
        ans, toks = cot(q, temperature=0.8)
        votes.append(ans); total += toks
    return Counter(votes).most_common(1)[0][0], total


def thinking(q):  # built-in reasoning mode (qwen3 thinks by default; hosted reasoning models always do)
    out, toks = ask(f"{q}\nGive the final answer as 'Answer: <number>'.", think=True)
    m = re.search(r"Answer:\s*\$?(-?[\d,.]+)", out)
    return (float(m.group(1).replace(",", "").rstrip(".")) if m else last_number(out)), toks

## 2. Run the comparison
`completion_tokens` includes hidden reasoning tokens for thinking models — that's what you pay for.

In [ ]:
def evaluate(method, name, problems=PROBLEMS):
    start, correct, tokens = time.perf_counter(), 0, 0
    for q, gold in problems:
        ans, toks = method(q)
        correct += ans is not None and abs(ans - gold) < 1e-6
        tokens += toks
    secs = time.perf_counter() - start
    print(f"{name:22} {correct}/{len(problems)} correct | {tokens / len(problems):6.0f} output tokens/problem | {secs / len(problems):5.1f} s/problem")
    return correct / len(problems)


results = {
    "direct": evaluate(direct, "direct answer"),
    "cot": evaluate(cot, "chain-of-thought"),
    "self-consistency": evaluate(self_consistency, "self-consistency k=5"),
    "thinking": evaluate(thinking, "built-in thinking"),
}

Typical pattern: direct < CoT ≤ self-consistency ≈ thinking, with tokens (= cost and latency) growing 10–100×.
Your numbers depend on the model; rerun with a hosted model to compare.

## 3. When is reasoning worth it?
| Use reasoning | Skip it |
|---|---|
| multi-step math/logic, planning, code, agent decisions, ambiguous analysis | classification, extraction, rewriting, simple Q&A, latency-critical UX |

Practical knobs:
- **Reasoning models** expose effort/budget controls (`reasoning_effort` low/medium/high on OpenAI-style APIs, thinking budget tokens on
  Anthropic/Gemini). Start low; raise only if your eval shows gains.
- Don't add "think step by step" to a reasoning model — it already does. Do give it clear goals and constraints.
- For non-reasoning models, CoT helps; ask for the reasoning **before** the answer (after is just rationalisation).
- Structured outputs + CoT: put a `reasoning: str` field *first* in the schema, then the answer field.

## Exercises
1. Rerun with a hosted model (Gemini Flash / gpt-oss on Groq). Which method wins, and at what token cost?
2. Write 10 harder problems where the direct answer fails even on a big model. Does thinking fix them?
3. Measure self-consistency with k = 1, 3, 5, 9. Plot accuracy vs tokens spent.
4. Add a `reasoning` field before `answer` in a Pydantic schema (05.03) and compare against the plain schema.

In [ ]:
# your code here